# Explore raw datasets

Walks `data/raw/<source>/` for each source listed in `data/sources.yaml` and renders:
- per-source image counts and class label counts (in the source's own indexing)
- a 4×4 sample grid of annotated images per source
- image-resolution distribution per source

Run this **before** `prepare_dataset.py` to spot imbalances and decide which sources to weight down.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
RAW = ROOT / 'data' / 'raw'
print('project root:', ROOT)
print('raw dir:', RAW, 'exists:', RAW.exists())

In [ ]:
import yaml
with open(ROOT / 'data' / 'sources.yaml') as f:
    cfg = yaml.safe_load(f)
for s in cfg['sources']:
    d = RAW / s['name']
    print(f"{s['name']:<28} {s['type']:<12} present={d.exists()}")

In [ ]:
from collections import Counter
import re
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

def find_pairs(root):
    pairs = []
    if not root.exists():
        return pairs
    for img in root.rglob('*'):
        if img.suffix.lower() not in IMG_EXTS or not img.is_file():
            continue
        for cand in [img.with_suffix('.txt'), img.parent.parent / 'labels' / (img.stem + '.txt')]:
            if cand.exists():
                pairs.append((img, cand))
                break
    return pairs

summary = []
for s in cfg['sources']:
    pairs = find_pairs(RAW / s['name'])
    label_idx = Counter()
    for _, lbl in pairs:
        with open(lbl, errors='ignore') as f:
            for line in f:
                parts = line.strip().split()
                if parts and parts[0].lstrip('-').isdigit():
                    label_idx[int(parts[0])] += 1
    summary.append({'source': s['name'], 'images': len(pairs), 'labels_by_idx': dict(label_idx)})
import pandas as pd
pd.DataFrame(summary)

In [ ]:
import random
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

def draw_boxes(img, lbl_path):
    im = img.copy().convert('RGB')
    if not lbl_path.exists():
        return im
    draw = ImageDraw.Draw(im)
    W, H = im.size
    palette = {0: 'red', 1: 'orange', 2: 'yellow', 3: 'magenta'}
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls = int(parts[0]); cx, cy, w, h = map(float, parts[1:5])
            x0 = (cx - w/2) * W; y0 = (cy - h/2) * H
            x1 = (cx + w/2) * W; y1 = (cy + h/2) * H
            draw.rectangle([x0, y0, x1, y1], outline=palette.get(cls, 'lime'), width=3)
    return im

for s in cfg['sources']:
    pairs = find_pairs(RAW / s['name'])
    if not pairs:
        continue
    sample = random.Random(42).sample(pairs, k=min(8, len(pairs)))
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for ax, (img_p, lbl_p) in zip(axes.flatten(), sample):
        try:
            with Image.open(img_p) as im:
                ax.imshow(draw_boxes(im, lbl_p))
        except Exception as e:
            ax.text(0.5, 0.5, str(e)[:40], ha='center')
        ax.set_title(img_p.name[:30], fontsize=8)
        ax.axis('off')
    fig.suptitle(s['name'], fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
from PIL import Image
rows = []
for s in cfg['sources']:
    for img, _ in find_pairs(RAW / s['name'])[:500]:
        try:
            with Image.open(img) as im:
                rows.append({'source': s['name'], 'w': im.width, 'h': im.height})
        except Exception:
            pass
df = pd.DataFrame(rows)
if not df.empty:
    df.groupby('source').agg(['mean','min','max','count']).round(0)